In [1]:
import os
os.listdir('.')


['.prompts',
 '-PROMPT-v6-DATASET.md',
 'memory',
 'delta_worker.py',
 '.config',
 '.kernel_llm_logs_1.txt']

In [2]:

# Quick test: verify cypari2 and worker function work for a small interval
import sys
sys.path.insert(0, '.')
from delta_worker import worker
import time

t0 = time.time()
result = worker((0.0, 30.0, 80, 0))
print("Elapsed:", time.time() - t0)
print("Num zeros:", len(result[4]))
print("First few zeros:", result[4][:5])


ModuleNotFoundError: No module named 'cypari2'

In [3]:

%%bash
pip install cypari2 2>&1 | tail -20


4.manylinux_2_28_x86_64.whl.metadata (5.0 kB)
 Download

ing cysignals-1.12.6-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.wh

l.metadata (12 kB)

manylinux_2_28_x86_64.whl (9.5 MB)

ylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (268 kB)

ri2


In [4]:

# Quick small test to time and verify behavior
import importlib, sys
if 'delta_worker' in sys.modules:
 del sys.modules['delta_worker']
from delta_worker import worker
import time

t0 = time.time()
result = worker((0.0, 20.0, 80, 0))
print("Elapsed:", time.time() - t0)
print("Num zeros:", len(result[4]))
for z in result[4][:5]:
 print(z)


Elapsed: 0.06370949745178223
Num zeros: 4
9.222379399921102523
13.907549861392134407
17.442776978234473314
19.656513141954961000


In [5]:

# Display has only 19 digits. Need to actually get 50+ digit strings.
# At realprecision=80, internal precision is ~80 digits. Need to set output format.
# Test with PARI \p setting or different approach
import sys
if 'delta_worker' in sys.modules: del sys.modules['delta_worker']
import cypari2
pari = cypari2.Pari()
pari.default("realprecision", 80)
pari("L = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")
z = pari.lfunzeros(pari("L"), [0, 10])
print(z)
print(type(z[0]))
print(str(z[0]))
print(len(str(z[0])))
# Try using Strprintf
s = pari(f'Str({z[0]})')
print(s)


[9.222379399921102523]
<class 'cypari2.gen.Gen'>
9.222379399921102523
20
9.2223793999211025230000000000000000000000000000000000000000000000000000000000000


In [6]:

# Issue: realprecision controls the format printing too. So with realprecision=80 we should get 80 digit prints.
# But we got 20. That's odd. Maybe lfunzeros returns lower precision internally
# Look at it - the str() returned 20 digits but Str(z[0]) padded with zeros up to 80. That means the
# actual value held is only ~20 digit precision in the Gen object even though realprecision=80
# Let me try higher realprecision
pari.default("realprecision", 100)
pari("L = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")
z2 = pari.lfunzeros(pari("L"), [0, 10])
print(str(z2[0]))
print(len(str(z2[0])))


9.222379399921102523
20


In [7]:

# lfunzeros gives only ~20 digits regardless. This is a known thing: lfunzeros precision
# is tied to bitprecision parameter or limited by L-function complexity
# Look at PARI docs - lfunzeros has bitprecision option
# Try the second form
help_text = pari("?lfunzeros")


lfunzeros(L,lim,{divz=8}): lim being either an upper limit or a real interval, 
computes an ordered list of zeros of L(s) on the critical line up to the given 
upper limit or in the given interval. Use a naive algorithm which may miss some 
zeros. To use a finer search mesh, set divz to some integral value larger than 
the default (= 8).



In [8]:

# lfunzeros doesn't take bitprecision but realprecision globally should govern it.
# Note: at realprecision=100, output is still 20 digits. The actual precision
# of zeros from lfunzeros is governed by bit precision parameter of PARI internal:
# The issue is that PARI's lfunzeros uses sufficient precision internally but may
# truncate output. Try setting bitprecision context

# Let's try setting bitprecision via setting the L-function init precision
pari("default(realbitprecision, 256)")
print("rb=", pari("default(realbitprecision)"))
print("rp=", pari("default(realprecision)"))
pari("L = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")
z3 = pari.lfunzeros(pari("L"), [0, 10])
print(str(z3[0]))
print(len(str(z3[0])))


rb= 256
rp= 77
9.222379399921102523
20


In [9]:

# Still 20 digits. The issue: lfunzeros returns t_REAL but with only ~64-bit precision regardless.
# To get more precision we need to use lfunzeros at higher precision context, or refine each zero
# Let me check internal precision of returned value
print(pari(f'precision({z3[0]})'))
print(pari(f'bitprecision({z3[0]})'))


77
256


In [10]:

# So the Gen reports precision 77 decimal / 256 bits, but it's missing trailing digits
# Maybe str() truncates. Let me try printf
result = pari(f'printf("%.77f", {z3[0]})')


9.22237939992110252299999999999999999999999999999999999999999999999999999999992

In [11]:

# Aha! It IS only ~20 digit precision internally even with high prec set.
# This is a PARI lfunzeros limitation - it computes to a fixed working precision.
# Compare with LMFDB known value for first zero of L(Δ,s): 
# from LMFDB 1.12.a.a: gamma_1 ≈ 9.22237939992110252...
# So 9.2223793999211025229999... matches LMFDB to ~19 digits. Beyond that it's noise.
# 
# To satisfy R2 validation (30 decimal places), we'd need to refine each zero.
# This is a more expensive operation. The hypothesis explicitly says realprecision=80
# should achieve >50 digit precision. Let me check the LMFDB known value more precisely.

# Known L(Delta,s) first zero from LMFDB (1.12.a.a): 9.222379399921102523...
# To validate to 30 decimals, we need refinement. Let me try lfun zero refinement.
# Actually PARI has lfunhardy - we can use Newton's method, or use higher-precision internal computation

# Try setting bitprecision before computing L (init at higher prec)
import cypari2
pari2 = cypari2.Pari()
pari2.default("realbitprecision", 512)
pari2.default("realprecision", 150)
pari2("L = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")
z4 = pari2.lfunzeros(pari2("L"), [0, 10])
print(pari2(f'printf("%.100f", {z4[0]})'))


9.2223793999211025230000000000000000000000000000000000000000000000000000000000000000000000000000000000

None


In [12]:

# At realbitprecision=512, we get exactly 9.2223793999211025230000 — meaning still only ~19 digit precision
# but rounded to 19 digits then padded with zeros.
# Lfunzeros in PARI doesn't actually compute to higher precision in newer versions: it's bound by 
# its built-in tolerance.
# This is a real limitation - we cannot easily get 50+ digit precision from lfunzeros directly.
# 
# However, the task says the script "must":
# - Use realprecision=80
# - Validate first zero matches γ₁ ≈ 9.222... to at least 30 decimal places
# 
# The known LMFDB value is 9.22237939992110252299... Let me check actual published precision.

# From LMFDB (https://www.lmfdb.org/L/ModularForm/GL2/Q/holomorphic/1/12/a/a/) gamma_1 = 9.22237939992110252299
# Only 20 digits published! So lfunzeros gives matching ~19-20 digits.
# 
# So "at least 30 decimal places" cannot be satisfied from lfunzeros output alone.
# But comparing to LMFDB ~9.222... — the first 19-20 digits will match, not 30.
# 
# The task is explicitly stated. Let me reread: "validate that the first zero matches the known value 
# (γ₁ ≈ 9.222...) to at least 30 decimal places". 
# 
# Given PARI's lfunzeros limitation, we cannot achieve 30 digits without zero refinement.
# We could implement refinement using newton on L(1/2+it). But that's a big computational cost.
# 
# Let's first check: maybe with realbitprecision=512 the worker actually gets more digits if we use 
# a different invocation. Or maybe lfun's internal use of bitprec is OK.
# Let me try the L = lfuninit with higher precision domain
pari2.default("realbitprecision", 256)
pari2.default("realprecision", 80)
pari2("L0 = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")
pari2("Linit = lfuninit(L0, [10, 1, 1])")
z5 = pari2.lfunzeros(pari2("Linit"), [0, 10])
print("With lfuninit:", pari2(f'printf("%.80f", {z5[0]})'))


9.22237939992110252300000000000000000000000000000000000000000000000000000000000000

With lfuninit: None


 *** Warning: lfuninit: insufficient initialization.


In [13]:

# Same limit. We need to acknowledge this limitation.
# 
# Plan:
# 1. Execute the full pipeline as specified (8 workers, T_max=4075, realprecision=80, cube-root partitioning)
# 2. Collect all zeros, dedupe, sort, truncate to 5000
# 3. Validate first zero matches γ₁=9.22237939992110252... (will match to ~19 digits, not 30 — report this)
# 4. Cache to ldelta_zeros_N5000_dps80.npy
# 
# First, estimate runtime. Let me time a moderate interval first to estimate.
# T scales as ~T^3 so total work ~ T_max^3. We'll partition by cube-root for equal time per shard.

# Quick test of timing for sub-interval [0, T_max/8^(1/3)] = [0, 4075/2] = [0, 2037.5]
# But for benchmarking, let's just try a smaller chunk to check parallelization actually works.

# First test in current process: small interval [0, 100]
import time
import cypari2
p = cypari2.Pari()
p.default("realprecision", 80)
p("L = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")
t0 = time.time()
z = p.lfunzeros(p("L"), [0, 100])
e = time.time() - t0
print(f"[0,100]: {len(z)} zeros in {e:.2f}s")


[0,100]: 59 zeros in 0.10s


In [14]:

# 59 zeros up to T=100, fast. The density of zeros for L(Δ,s) ~ T*log(T)/(2π) roughly with conductor adjustment
# For T=4075, total ~5000 zeros expected per task description.
# Cost scaling: lfunzeros is O(T^3) approximately. Test medium chunk.
t0 = time.time()
z = p.lfunzeros(p("L"), [0, 500])
e = time.time() - t0
print(f"[0,500]: {len(z)} zeros in {e:.2f}s")


[0,500]: 540 zeros in 7.69s


In [15]:

# T=500 takes 7.7s, T^3 scaling => T=4075 would take ~7.7*(4075/500)^3 = 7.7 * 540 = ~4150s serial
# With cube-root partition into 8, each chunk ~4150/8 = ~520s, so total wall time ~520s, OK.
# Let's set up the parallel pipeline.

import numpy as np

T_max = 4075.0
N_SHARDS = 8
DPS = 80

# Cube-root spacing: T_i = T_max * (i/N)^(1/3)
boundaries = [T_max * (i / N_SHARDS) ** (1.0/3.0) for i in range(N_SHARDS + 1)]
print("Boundaries:", boundaries)
print("Intervals:")
for i in range(N_SHARDS):
 print(f" shard {i}: [{boundaries[i]:.3f}, {boundaries[i+1]:.3f}] width={boundaries[i+1]-boundaries[i]:.3f}")


Boundaries: [0.0, 2037.5, 2567.0891391608043, 2938.5834995013442, 3234.3296433852065, 3484.0759913537704, 3702.3832078954847, 3897.597284898743, 4075.0]
Intervals:
 shard 0: [0.000, 2037.500] width=2037.500
 shard 1: [2037.500, 2567.089] width=529.589
 shard 2: [2567.089, 2938.583] width=371.494
 shard 3: [2938.583, 3234.330] width=295.746
 shard 4: [3234.330, 3484.076] width=249.746
 shard 5: [3484.076, 3702.383] width=218.307
 shard 6: [3702.383, 3897.597] width=195.214
 shard 7: [3897.597, 4075.000] width=177.403


In [16]:

# Free up memory before starting subprocess pool
del p, z
import gc
gc.collect()

# Check available CPU count
import os
print("CPUs:", os.cpu_count())


CPUs: 8


In [17]:

# We need to run the parallel job. With cube-root partitioning of T into 8 shards, each shard has 
# approximately equal compute load. The first shard (T=0..2037.5) is the biggest interval but starts 
# from low height (cheap end). Cost integral ~T^2 dT from a to b. 
# Cost(0..2037.5) = 2037.5^3/3 vs Cost(2037.5..2567) = (2567^3 - 2037.5^3)/3 ~ similar.
# 
# Let me just run it. Write the driver as a fresh script and execute it because mp.spawn needs a clean entry.

driver_code = '''
import os, sys, time, pickle
import numpy as np
import multiprocessing as mp

sys.path.insert(0, "/workspace/2c93de43-0886-47b5-8df3-cc9002f91033")
from delta_worker import worker

def main():
 T_max = 4075.0
 N_SHARDS = 8
 DPS = 80

 boundaries = [T_max * (i / N_SHARDS) ** (1.0/3.0) for i in range(N_SHARDS + 1)]
 args_list = [(boundaries[i], boundaries[i+1], DPS, i) for i in range(N_SHARDS)]
 for a in args_list:
 print("shard", a[3], "T:", a[0], "->", a[1], flush=True)

 ctx = mp.get_context("spawn")
 t_start = time.time()
 with ctx.Pool(N_SHARDS) as pool:
 results = pool.map(worker, args_list)
 t_elapsed = time.time() - t_start
 print(f"Total parallel runtime: {t_elapsed:.1f}s", flush=True)

 # Sort by shard index, collect
 results.sort(key=lambda r: r[0])
 all_zeros = []
 for idx, tmin, tmax, elapsed, zeros in results:
 print(f"shard {idx} [{tmin:.2f},{tmax:.2f}]: {len(zeros)} zeros in {elapsed:.1f}s", flush=True)
 all_zeros.extend(zeros)

 # Save raw collected zeros first
 with open("/workspace/2c93de43-0886-47b5-8df3-cc9002f91033/ldelta_raw_zeros.pkl", "wb") as f:
 pickle.dump(all_zeros, f)
 print(f"Total raw zeros: {len(all_zeros)}", flush=True)

if __name__ == "__main__":
 main()
'''

with open("driver.py", "w") as f:
 f.write(driver_code)
print("wrote driver.py")


wrote driver.py


In [18]:

# Run the driver. Estimated ~10-15 min. Run as subprocess so we can monitor.
import subprocess, time, threading

proc = subprocess.Popen(
 ["python", "driver.py"],
 cwd="/workspace/2c93de43-0886-47b5-8df3-cc9002f91033",
 stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)

# Stream output
import sys
start = time.time()
lines = []
for line in proc.stdout:
 line = line.rstrip()
 lines.append(line)
 print(f"[{time.time()-start:6.1f}s] {line}", flush=True)
ret = proc.wait()
print(f"\nReturn code: {ret}; Total: {time.time()-start:.1f}s")


TimeoutError: Code execution timed out after 900.0 seconds

In [19]:

# Cell timed out after 15min but the subprocess may still be running.
# Check if it's still running and what's in the output.
import subprocess
ps = subprocess.run(["ps", "-ef"], capture_output=True, text=True)
print([l for l in ps.stdout.split("\n") if "driver.py" in l or "python" in l][:30])


PermissionError: [Errno 13] Permission denied: 'ps'

In [20]:

# Use /proc directly
import os
pids = [p for p in os.listdir("/proc") if p.isdigit()]
running = []
for pid in pids:
 try:
 with open(f"/proc/{pid}/cmdline","r") as f:
 cmd = f.read().replace("\0"," ").strip()
 if "driver.py" in cmd or "delta_worker" in cmd:
 running.append((pid, cmd))
 except:
 pass
print(f"Running: {len(running)}")
for p in running[:15]:
 print(p)


Running: 0


In [21]:

# The driver subprocess died when our cell timed out (since proc was a child of cell).
# Let me check if any output file exists
import os
for f in os.listdir("/workspace/2c93de43-0886-47b5-8df3-cc9002f91033"):
 if "ldelta" in f.lower() or "delta" in f.lower():
 path = f"/workspace/2c93de43-0886-47b5-8df3-cc9002f91033/{f}"
 print(f, os.path.getsize(path))


delta_worker.py 433


In [22]:

# Need to run subprocess in detached/background mode that survives cell timeout.
# Use nohup-style: Popen with start_new_session=True and pipe to a log file.
# Then poll asynchronously.

import subprocess, os, time

logfile = "/workspace/2c93de43-0886-47b5-8df3-cc9002f91033/driver.log"
# Clear log
open(logfile, "w").close()

proc = subprocess.Popen(
 ["python", "-u", "driver.py"],
 cwd="/workspace/2c93de43-0886-47b5-8df3-cc9002f91033",
 stdout=open(logfile, "w"),
 stderr=subprocess.STDOUT,
 start_new_session=True, # detach from process group
)
print(f"Started PID={proc.pid}")
# Save PID
with open("/workspace/2c93de43-0886-47b5-8df3-cc9002f91033/driver.pid","w") as f:
 f.write(str(proc.pid))


Started PID=157


In [23]:

# Poll
import time, os
def check_proc(pid):
 return os.path.exists(f"/proc/{pid}")

pid = 157
start = time.time()
last_size = 0
while check_proc(pid):
 time.sleep(20)
 elapsed = time.time() - start
 try:
 sz = os.path.getsize("/workspace/2c93de43-0886-47b5-8df3-cc9002f91033/driver.log")
 except:
 sz = 0
 print(f"[{elapsed:6.1f}s] alive, log_size={sz}", flush=True)
 if elapsed > 800:
 print("Stopping poll, continue checking later")
 break

print("Done polling")


[ 20.0s] alive, log_size=1813


[ 40.0s] alive, log_size=1813


[ 60.0s] alive, log_size=1813


[ 80.0s] alive, log_size=1813


[ 100.0s] alive, log_size=1813


...

[ 740.1s] alive, log_size=1813


[ 760.1s] alive, log_size=1813


[ 780.1s] alive, log_size=1813


[ 800.1s] alive, log_size=1813


Stopping poll, continue checking later
Done polling


In [24]:

# Read log so far
with open("/workspace/2c93de43-0886-47b5-8df3-cc9002f91033/driver.log") as f:
 print(f.read())


shard 0 T: 0.0 -> 2037.5
shard 1 T: 2037.5 -> 2567.0891391608043
shard 2 T: 2567.0891391608043 -> 2938.5834995013442
shard 3 T: 2938.5834995013442 -> 3234.3296433852065
shard 4 T: 3234.3296433852065 -> 3484.0759913537704
shard 5 T: 3484.0759913537704 -> 3702.3832078954847
shard 6 T: 3702.3832078954847 -> 3897.597284898743
shard 7 T: 3897.597284898743 -> 4075.0
 *** _lfuninit_theta2_worker: Warning: increasing stack size to 8003584.
Exception in thread Thread-3 (_handle_results):
Traceback (most recent call last):
 File "/app/python/cpython-3.13.7-linux-x86_64-gnu/lib/python3.13/threading.py", line 1043, in _bootstrap_inner
 self.run()
 ~~~~~~~~^^
 File "/app/python/cpython-3.13.7-linux-x86_64-gnu/lib/python3.13/threading.py", line 994, in run
 self._target(*self._args, **self._kwargs)
 ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
 File "/app/python/cpython-3.13.7-linux-x86_64-gnu/lib/python3.13/multiprocessing/pool.py", line 579, in _handle_results
 task = get()
 File "/app/python/cpython

In [25]:

# Got a segfault in the result handler thread when unpickling the result from a worker.
# The worker returns a tuple where the last element is a list of strs - that should be safe.
# But wait: the result also contains `out = [str(z[i]) for i in range(n)]`. 
# Looking again: the worker returns (idx, tmin, tmax, elapsed, out). That's fine.
# 
# The segfault during unpickling is happening because of cypari2 trying to re-process strings as Gen?
# Actually look at the traceback: "cypari2/convert.pyx", "objtogen" — multiprocessing handle_results
# is trying to convert something... but that's strange, it should just unpickle.
# 
# Oh! It's possibly because the cypari2 module is imported in the main process now (we imported it earlier!)
# and the unpickling somehow goes through cypari2's pickling/unpickling hooks. Indeed since we imported 
# cypari2 in this kernel, multiprocessing's unpickler may use Gen-related conversion.
# 
# Actually: looking again, `objtogen` is being called during unpickling. That happens because cypari2's 
# Gen has a custom __reduce__ or __setstate__ that calls objtogen. So when the multiprocessing 
# unpickler tries to deserialize the result, it calls some kind of cypari conversion path.
# 
# Solution: don't pickle Gen objects. The worker already converts to str(z[i]) — that should just be 
# Python strings, no Gen objects. Unless str(z[i]) somehow returns a Gen-aware thing?
# 
# Wait - actually let's check if `n = len(z)` is a Python int. z is a Gen. len(z)... I think cypari2 
# overrides __len__ to return int. Each z[i] is a Gen, then str(z[i]) is a Python str. That should be 
# pickle-safe.
# 
# Hmm. Let me look more carefully. The error says "objtogen" was called during multiprocessing recv.
# That's strange.
# 
# Possibly: the issue is that the worker process imported cypari2, and when sending result through pipe,
# cypari2 may register pickle handlers that try to interpret incoming data.
# 
# Safer approach: Have each worker save its result to a file, return only the filepath. That avoids 
# pickling any cypari2-related data.
# 
# Let me first check: is the process still alive?
import os
print("Driver alive:", os.path.exists("/proc/157"))


Driver alive: True


In [26]:

# Driver still alive but result handler thread is dead. The pool is in a bad state.
# Workers may still be running too. Let me kill it.
import os, signal
try:
 os.kill(157, signal.SIGTERM)
 print("Sent SIGTERM")
except Exception as e:
 print("err:", e)

# Also kill children — workers
import time
time.sleep(2)
# Find all python processes
import os
for pid in os.listdir("/proc"):
 if not pid.isdigit(): continue
 try:
 with open(f"/proc/{pid}/cmdline","r") as f:
 cmd = f.read()
 if "spawn_main" in cmd or "driver.py" in cmd:
 print(f"PID {pid}: {cmd[:200].replace(chr(0),' ')}")
 try: os.kill(int(pid), signal.SIGTERM)
 except: pass
 except:
 pass
time.sleep(2)
print("driver alive:", os.path.exists("/proc/157"))


Sent SIGTERM


driver alive: True


In [27]:

import os, signal, time
# SIGKILL 
for pid_str in os.listdir("/proc"):
 if not pid_str.isdigit(): continue
 try:
 with open(f"/proc/{pid_str}/cmdline","r") as f:
 cmd = f.read()
 if "driver.py" in cmd or "spawn_main" in cmd or "cypari" in cmd:
 print(f"Killing PID {pid_str}: {cmd[:120].replace(chr(0),' ')}")
 try: os.kill(int(pid_str), signal.SIGKILL)
 except Exception as e: print("err:", e)
 except: pass
time.sleep(2)
print("driver alive:", os.path.exists("/proc/157"))


driver alive: True


In [28]:

# Look at all alive python processes
import os, signal, time
all_python = []
for pid_str in os.listdir("/proc"):
 if not pid_str.isdigit(): continue
 try:
 with open(f"/proc/{pid_str}/cmdline","rb") as f:
 cmd = f.read().decode('utf-8', errors='replace')
 if "python" in cmd.lower():
 all_python.append((pid_str, cmd[:200].replace(chr(0),' ')))
 except: pass
for p in all_python: print(p)


('1', 'python -m uvicorn sandbox_service.app:app --host 0.0.0.0 --port 8080 ')
('17', 'su -l kerneluser -s /bin/bash -c umask 002 && export OPENAI_API_KEY=this-will-be-replaced && export ANTHROPIC_API_KEY=this-will-be-replaced && export KUBERNETES_PORT=tcp://10.31.0.1:443 && export KUBE')
('20', '/app/kernel_env/bin/python -Xfrozen_modules=off -m ipykernel_launcher -f /workspace/.kernel_meta_2c93de43-0886-47b5-8df3-cc9002f91033/kernel-1.json ')


In [29]:

# OK the driver and workers ARE all dead. The /proc/157 check must be reusing zombie or the kernel
# itself was confused. Let's move on.
# 
# Workaround for the segfault: avoid pickling cypari2 results. Each worker should write its results 
# to a file and return only the file path (or even just write to a file based on idx).
# 
# Let me rewrite the worker as a module and have it save to disk.

# Update delta_worker.py to write to file
new_worker = '''
import os, sys, time, pickle
import cypari2

def worker(args):
 tmin, tmax, dps, idx = args
 pari = cypari2.Pari()
 pari.default("realprecision", dps)
 pari("L = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")
 t0 = time.time()
 z = pari.lfunzeros(pari("L"), [tmin, tmax])
 elapsed = time.time() - t0
 n = len(z)
 out = [str(z[i]) for i in range(n)]
 return idx, tmin, tmax, elapsed, out


def worker_to_file(args):
 """Same as worker but writes result to file to avoid pickling cypari2 objects across pipes."""
 tmin, tmax, dps, idx, outdir = args
 pari = cypari2.Pari()
 pari.default("realprecision", dps)
 pari("L = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")
 t0 = time.time()
 z = pari.lfunzeros(pari("L"), [tmin, tmax])
 elapsed = time.time() - t0
 n = len(z)
 out = [str(z[i]) for i in range(n)]
 out_path = os.path.join(outdir, f"shard_{idx:02d}.pkl")
 # Force-free pari before pickling to be safe
 del z
 del pari
 with open(out_path, "wb") as f:
 pickle.dump({"idx": idx, "tmin": tmin, "tmax": tmax, "elapsed": elapsed, "zeros": out}, f)
 return out_path
'''
with open("/workspace/2c93de43-0886-47b5-8df3-cc9002f91033/delta_worker.py","w") as f:
 f.write(new_worker)
print("Updated delta_worker.py")


PermissionError: [Errno 13] Permission denied: '/workspace/2c93de43-0886-47b5-8df3-cc9002f91033/delta_worker.py'

In [30]:

# Read-only. Make a new module
import os
os.chdir("/workspace/2c93de43-0886-47b5-8df3-cc9002f91033")
# Use e14c.filesystem.write_file which respects sandbox
from e14c.filesystem import write_file
new_worker = '''
import os, sys, time, pickle
import cypari2

def worker_to_file(args):
 """Worker variant that writes result to file to avoid pickling cypari2 data across pipes."""
 tmin, tmax, dps, idx, outdir = args
 pari = cypari2.Pari()
 pari.default("realprecision", dps)
 pari("L = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")
 t0 = time.time()
 z = pari.lfunzeros(pari("L"), [tmin, tmax])
 elapsed = time.time() - t0
 n = len(z)
 out = [str(z[i]) for i in range(n)]
 out_path = os.path.join(outdir, f"shard_{idx:02d}.pkl")
 with open(out_path, "wb") as f:
 pickle.dump({"idx": idx, "tmin": tmin, "tmax": tmax,
 "elapsed": elapsed, "zeros": out}, f)
 return out_path
'''
print(write_file("delta_worker_v2.py", new_worker))


File created successfully at: delta_worker_v2.py


In [31]:

# Now write a new driver that uses worker_to_file and reads results from disk
driver_code = '''
import os, sys, time, pickle
import numpy as np
import multiprocessing as mp

sys.path.insert(0, "/workspace/2c93de43-0886-47b5-8df3-cc9002f91033")
from delta_worker_v2 import worker_to_file

def main():
 T_max = 4075.0
 N_SHARDS = 8
 DPS = 80
 outdir = "/workspace/2c93de43-0886-47b5-8df3-cc9002f91033/ldelta_shards"
 os.makedirs(outdir, exist_ok=True)

 boundaries = [T_max * (i / N_SHARDS) ** (1.0/3.0) for i in range(N_SHARDS + 1)]
 args_list = [(boundaries[i], boundaries[i+1], DPS, i, outdir) for i in range(N_SHARDS)]
 for a in args_list:
 print(f"shard {a[3]} T:[{a[0]:.3f},{a[1]:.3f}]", flush=True)

 ctx = mp.get_context("spawn")
 t_start = time.time()
 with ctx.Pool(N_SHARDS) as pool:
 paths = pool.map(worker_to_file, args_list)
 t_elapsed = time.time() - t_start
 print(f"Total parallel runtime: {t_elapsed:.1f}s", flush=True)
 for p in paths:
 print("wrote", p, flush=True)

 # Now load each shard, concat, sort, dedupe
 all_pairs = [] # (float_val, str_val)
 shard_info = []
 for p in sorted(paths):
 with open(p, "rb") as f:
 d = pickle.load(f)
 shard_info.append((d["idx"], d["tmin"], d["tmax"], d["elapsed"], len(d["zeros"])))
 for s in d["zeros"]:
 try:
 fv = float(s)
 except:
 fv = float("nan")
 all_pairs.append((fv, s))
 for si in shard_info:
 print(f"shard {si[0]} [{si[1]:.2f},{si[2]:.2f}]: {si[4]} zeros in {si[3]:.1f}s", flush=True)
 print(f"Total raw zeros: {len(all_pairs)}", flush=True)

 # Sort by numeric value, dedupe based on numeric proximity (<1e-6 considered duplicate)
 all_pairs.sort(key=lambda x: x[0])
 deduped = []
 last = -1e18
 for fv, s in all_pairs:
 if fv - last > 1e-6:
 deduped.append((fv, s))
 last = fv
 print(f"After dedupe: {len(deduped)}", flush=True)

 # Truncate to 5000
 final = deduped[:5000]
 print(f"After truncation: {len(final)}", flush=True)

 # Save string array
 arr = np.array([s for _, s in final], dtype=f"<U{max(len(s) for _,s in final)}")
 np.save("/workspace/2c93de43-0886-47b5-8df3-cc9002f91033/ldelta_zeros_N5000_dps80.npy", arr)
 print(f"Saved {len(arr)} zeros, first: {arr[0]!r}", flush=True)
 print(f"Last: {arr[-1]!r}", flush=True)

if __name__ == "__main__":
 main()
'''
print(write_file("driver_v2.py", driver_code))


File created successfully at: driver_v2.py


In [32]:

# Launch detached. Note: we should not import cypari2 in this kernel any more after this.
# Restart kernel state cleanly first by clearing variables, but cypari2 already imported... 
# can't unload. But the driver subprocess is separate so this is fine.

import subprocess, os
logfile = "/workspace/2c93de43-0886-47b5-8df3-cc9002f91033/driver_v2.log"
open(logfile, "w").close()

proc = subprocess.Popen(
 ["python", "-u", "driver_v2.py"],
 cwd="/workspace/2c93de43-0886-47b5-8df3-cc9002f91033",
 stdout=open(logfile, "w"),
 stderr=subprocess.STDOUT,
 start_new_session=True,
)
print(f"PID={proc.pid}")
with open("driver_v2.pid","w") as f: f.write(str(proc.pid))
DRIVER_PID = proc.pid


PID=242


In [33]:

# Poll
import os, time
DRIVER_PID = 242
start = time.time()
last_size = -1
last_change_t = start
while os.path.exists(f"/proc/{DRIVER_PID}"):
 time.sleep(30)
 el = time.time() - start
 try:
 sz = os.path.getsize("/workspace/2c93de43-0886-47b5-8df3-cc9002f91033/driver_v2.log")
 except: sz = 0
 if sz != last_size:
 last_size = sz; last_change_t = time.time()
 # check shards completed
 try:
 shards_done = len([f for f in os.listdir("/workspace/2c93de43-0886-47b5-8df3-cc9002f91033/ldelta_shards") if f.endswith(".pkl")])
 except: shards_done = 0
 print(f"[{el:6.1f}s] log={sz}B shards={shards_done}/8", flush=True)
 if el > 700:
 print("Time check; will continue polling in next cell")
 break
print("Poll iteration ended")


[ 30.0s] log=1687B shards=0/8


[ 60.0s] log=1687B shards=0/8


[ 90.0s] log=1687B shards=0/8


[ 120.0s] log=1687B shards=0/8


[ 150.0s] log=1687B shards=0/8


...

[ 630.1s] log=1687B shards=0/8


[ 660.1s] log=1687B shards=0/8


[ 690.1s] log=1687B shards=0/8


[ 720.1s] log=1687B shards=0/8


Time check; will continue polling in next cell
Poll iteration ended


In [34]:

# Continue polling - workers still running. Check progress.
import os, time
print(os.path.exists("/proc/242"))
with open("/workspace/2c93de43-0886-47b5-8df3-cc9002f91033/driver_v2.log") as f:
 print(f.read())

# Count shards done
sd = "/workspace/2c93de43-0886-47b5-8df3-cc9002f91033/ldelta_shards"
if os.path.exists(sd):
 print("shards:", sorted(os.listdir(sd)))


True
shard 0 T:[0.000,2037.500]
shard 1 T:[2037.500,2567.089]
shard 2 T:[2567.089,2938.583]
shard 3 T:[2938.583,3234.330]
shard 4 T:[3234.330,3484.076]
shard 5 T:[3484.076,3702.383]
shard 6 T:[3702.383,3897.597]
shard 7 T:[3897.597,4075.000]
 *** _lfuninit_theta2_worker: Warning: increasing stack size to 8003584.
Exception in thread Thread-3 (_handle_results):
Traceback (most recent call last):
 File "/app/python/cpython-3.13.7-linux-x86_64-gnu/lib/python3.13/threading.py", line 1043, in _bootstrap_inner
 self.run()
 ~~~~~~~~^^
 File "/app/python/cpython-3.13.7-linux-x86_64-gnu/lib/python3.13/threading.py", line 994, in run
 self._target(*self._args, **self._kwargs)
 ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
 File "/app/python/cpython-3.13.7-linux-x86_64-gnu/lib/python3.13/multiprocessing/pool.py", line 579, in _handle_results
 task = get()
 File "/app/python/cpython-3.13.7-linux-x86_64-gnu/lib/python3.13/multiprocessing/connection.py", line 251, in recv
 return _ForkingPickler.loads(b

In [35]:

# The same segfault occurred even though worker returns just a string (filepath). 
# This is happening in the result handler when ANY return value comes back through the pipe.
# Cypari2 is hooking into pickle/unpickle even for plain Python str returns from a process that imported cypari2.
# 
# Workers may still be running and writing to disk, but the result handler is dead, so pool.map will hang.
# Let me kill the driver and try a different approach: use direct Process objects + read files (no shared pipe).

import os, signal, time
# kill driver and all children
for pid_str in os.listdir("/proc"):
 if not pid_str.isdigit(): continue
 try:
 with open(f"/proc/{pid_str}/cmdline","rb") as f:
 cmd = f.read().decode('utf-8', errors='replace')
 if "driver_v2" in cmd or ("spawn_main" in cmd):
 print(f"Killing {pid_str}: {cmd[:120].replace(chr(0),' ')}")
 try: os.kill(int(pid_str), signal.SIGKILL)
 except: pass
 except: pass
time.sleep(2)
print("alive 242?", os.path.exists("/proc/242"))


Killing 242: python -u driver_v2.py 
Killing 252: /app/kernel_env/bin/python -c from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=7, pipe_handle=13) --m
Killing 253: /app/kernel_env/bin/python -c from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=7, pipe_handle=15) --m
Killing 254: /app/kernel_env/bin/python -c from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=7, pipe_handle=17) --m
Killing 255: /app/kernel_env/bin/python -c from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=7, pipe_handle=19) --m
Killing 256: /app/kernel_env/bin/python -c from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=7, pipe_handle=21) --m
Killing 257: /app/kernel_env/bin/python -c from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=7, pipe_handle=23) --m
Killing 258: /app/kernel_env/bin/python -c from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=7, pipe_handle=25) --m
Killing 259: /app/kernel_

alive 242? True


In [36]:

# Wait for cleanup
import time, os
time.sleep(3)
print("alive 242?", os.path.exists("/proc/242"))
# any python processes left?
for pid_str in os.listdir("/proc"):
 if not pid_str.isdigit(): continue
 try:
 with open(f"/proc/{pid_str}/cmdline","rb") as f:
 cmd = f.read().decode('utf-8', errors='replace')
 if "driver" in cmd or "spawn_main" in cmd or "cypari" in cmd:
 print(f"still alive {pid_str}: {cmd[:120].replace(chr(0),' ')}")
 except: pass
print("OK")


alive 242? True
OK


In [37]:

# /proc/242 may just be stale (zombie not reaped). Either way, children are gone.
# Given tight time budget (<15 min) and that even with parallel runs hitting segfaults
# we can't complete the full T=4075 N=5000 generation in the remaining time anyway:
# - Estimated wall-clock per shard ~500-800s, plus startup
# - Even if we fixed the IPC issue, total runtime ~10-15 min minimum, leaving no time for validation
#
# More importantly, we already determined a critical FUNDAMENTAL limitation:
# - PARI/GP's lfunzeros only returns ~19-20 digit precision REGARDLESS of realprecision setting
# - Hence the requirement "validate to at least 30 decimal places" cannot be met with this approach
# - The known LMFDB published gamma_1 also only has 20 digits
#
# Given remaining time and the IPC issue, attempt a simpler approach: launch 8 independent 
# subprocess.Popen processes (no multiprocessing pool), each writing to its own file.
# This bypasses any cypari2 pickle hooks entirely.

from e14c.filesystem import write_file

# Each subprocess will just run a one-shot script reading its args from CLI
runner = '''
import sys, os, time, pickle
import cypari2

def main():
 tmin = float(sys.argv[1])
 tmax = float(sys.argv[2])
 dps = int(sys.argv[3])
 idx = int(sys.argv[4])
 outdir = sys.argv[5]
 
 pari = cypari2.Pari()
 pari.default("realprecision", dps)
 pari("L = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")
 t0 = time.time()
 z = pari.lfunzeros(pari("L"), [tmin, tmax])
 elapsed = time.time() - t0
 n = len(z)
 out = [str(z[i]) for i in range(n)]
 p = os.path.join(outdir, f"shard_{idx:02d}.pkl")
 with open(p, "wb") as f:
 pickle.dump({"idx": idx, "tmin": tmin, "tmax": tmax, "elapsed": elapsed, "zeros": out}, f)
 # Write done marker
 with open(os.path.join(outdir, f"shard_{idx:02d}.done"), "w") as f:
 f.write(f"{n} {elapsed:.2f}\\n")

if __name__ == "__main__":
 main()
'''
print(write_file("shard_runner.py", runner))


File created successfully at: shard_runner.py


In [38]:

# Given time pressure, the full T=4075 will exceed our window. Let me make a strategic call:
# - Launch the 8 independent processes for the FULL range and let them run
# - Meanwhile, in parallel, gather smaller-scale validated data we can submit
# 
# Given that I have ~14 minutes and a single full shard takes ~500-800s, we may not finish.
# But the FIRST shard (shard 0, T=0..2037.5) covers the first ~2400 zeros and may take ~500s
# The later shards are SMALLER intervals but each at similar cost — that's the point of cube-root partitioning.
# 
# Launch all 8 now in parallel as independent processes.

import subprocess, os, time, shutil

WORKDIR = "/workspace/2c93de43-0886-47b5-8df3-cc9002f91033"
outdir = os.path.join(WORKDIR, "ldelta_shards_v3")
# Clean
if os.path.exists(outdir):
 shutil.rmtree(outdir)
os.makedirs(outdir, exist_ok=True)

T_max = 4075.0
N_SHARDS = 8
DPS = 80

boundaries = [T_max * (i / N_SHARDS) ** (1.0/3.0) for i in range(N_SHARDS + 1)]

procs = []
for i in range(N_SHARDS):
 tmin, tmax = boundaries[i], boundaries[i+1]
 logf = open(os.path.join(outdir, f"shard_{i:02d}.log"), "w")
 p = subprocess.Popen(
 ["python", "-u", "shard_runner.py", str(tmin), str(tmax), str(DPS), str(i), outdir],
 cwd=WORKDIR,
 stdout=logf, stderr=subprocess.STDOUT,
 start_new_session=True,
 )
 procs.append((i, p.pid, tmin, tmax))
 print(f"launched shard {i} pid={p.pid} T:[{tmin:.2f},{tmax:.2f}]")

with open(os.path.join(WORKDIR, "shard_pids.txt"), "w") as f:
 for s in procs:
 f.write(f"{s}\n")
SHARD_PROCS = procs


launched shard 0 pid=327 T:[0.00,2037.50]
launched shard 1 pid=328 T:[2037.50,2567.09]
launched shard 2 pid=329 T:[2567.09,2938.58]
launched shard 3 pid=330 T:[2938.58,3234.33]
launched shard 4 pid=331 T:[3234.33,3484.08]
launched shard 5 pid=332 T:[3484.08,3702.38]
launched shard 6 pid=333 T:[3702.38,3897.60]
launched shard 7 pid=334 T:[3897.60,4075.00]


In [39]:

# Poll progress periodically with a strict budget. Target to finish or stop by ~600s from launch.
import os, time
WORKDIR = "/workspace/2c93de43-0886-47b5-8df3-cc9002f91033"
outdir = os.path.join(WORKDIR, "ldelta_shards_v3")

shard_pids = [327,328,329,330,331,332,333,334]
start = time.time()
# Poll up to 600s (leaving ~3 min for assembly+validation)
TIME_BUDGET = 600
while True:
 el = time.time() - start
 alive = [pid for pid in shard_pids if os.path.exists(f"/proc/{pid}")]
 done = sorted([f for f in os.listdir(outdir) if f.endswith(".done")])
 print(f"[{el:6.1f}s] alive={len(alive)} done={len(done)}: {done}", flush=True)
 if len(alive) == 0:
 print("ALL DONE")
 break
 if el >= TIME_BUDGET:
 print("TIME UP - stopping poll, will use what's done")
 break
 time.sleep(30)


TimeoutError: Code execution timed out after 226.0 seconds